# Footfall Insights Notebook (NCDO and Octophin)

# Step 1: Data Cleaning

In [49]:
pip install matplotlib geopandas numpy scipy folium plotly

Note: you may need to restart the kernel to use updated packages.


In [50]:
#Import packages
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

The daily footfall 2019-2024 datasets for the different Bradford locations are loaded, concatenated and cleaned.

In [ ]:
import glob
import os

#Folder with containing all areas
folder_path = r"Footfall data 2019-2024"

#Find all CSVs in all subfolders
area_folders = glob.glob(os.path.join(folder_path, '*'))

#Define file names
custom_name = ['footfall']

#Dict to store combined dataframes by file type
combined = {name: pd.DataFrame() for name in custom_name}

for area_path in area_folders:
    #Get area name
    area_name = os.path.basename(area_path)
    
    csv_files = sorted(glob.glob(os.path.join(area_path, '*.csv')))
    
    #Loop over CSVs and assign new name
    for i, file in enumerate (csv_files):
        df = pd.read_csv(file, encoding='cp1252')
        
        #Add area column
        df['region'] = area_name
        
        #Get the new names
        new_name = custom_name[i]
        
        #Concatenate into 
        combined[new_name] = pd.concat([combined[new_name], df], ignore_index=True)

In [52]:
#Extract into dataframe
df_footfall = combined['footfall']

In [53]:
df_footfall.head()

,area,Interval,Date,Total Footfall,Average Footfall,Total Visiting,Average Visiting,Total Passing through,Average Passing through,Event took place,region,Unnamed: 9,Event Name,Event name
0,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,01/01/2019,844612.85,2307.69,408949.1487,1117.349636,435663.7013,1190.340364,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
1,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,07/01/2019,771831.49,2108.83,373709.4821,1021.064542,398122.0079,1087.765458,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
2,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,14/01/2019,955770.21,2611.39,462769.9113,1264.396720,493000.2987,1346.993280,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
3,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,21/01/2019,853455.73,2331.85,413230.7414,1129.047554,440224.9886,1202.802446,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
4,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,28/01/2019,839562.82,2293.89,406503.9982,1110.667879,433058.8218,1183.222121,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN


In [54]:
df_footfall['Event took place'].value_counts()

Event took place
YES    2938
Name: count, dtype: int64

In [55]:
#Only keep columns interested in
df_footfall = df_footfall[['area', 'Interval', 'Date', 'Total Footfall']]
df_footfall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38714 entries, 0 to 38713
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   area            38714 non-null  object 
 1   Interval        38714 non-null  object 
 2   Date            38714 non-null  object 
 3   Total Footfall  38714 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [56]:
df_footfall['Interval'].unique()

array(['dayOfWeek', 'daily', 'weekly', 'monthly'], dtype=object)

In [57]:
#Keep only the daily intervals
df_footfall = df_footfall[df_footfall['Interval'] == 'daily']

#Drop the interval column as the dataset now only contains daily data
df_footfall = df_footfall.drop(columns=['Interval'])

In [58]:
df_footfall['Date'] = pd.to_datetime(df_footfall['Date'], format='mixed', dayfirst= True)
df_footfall = df_footfall.rename(columns={'Date':'datestamp'})

In [59]:
#Rename total footfall column
df_footfall = df_footfall.rename(columns={'Total Footfall': 'estimated_actual_footfall'})

In [60]:
#Check the date range for each area
data_range_per_area = (
    df_footfall
    .groupby('area')['datestamp']
    .agg(min_date='min', max_date='max')
    .reset_index()
)

print(data_range_per_area)

                                                 area   min_date   max_date
0                                BD WALL - Wayfinders 2019-01-01 2026-01-17
1                      BD WALLS: Come on in my friend 2019-01-10 2025-01-01
2                               BD Walls - The Portal 2019-01-07 2024-12-09
3                     BD Walls : Serving the district 2019-01-07 2024-12-30
4                                      BD Walls Roots 2019-01-11 2024-10-01
5                                             Baildon 2019-01-07 2024-11-12
6   Balancing Art - Coates Street, Bradford Foyer,... 2019-02-11 2024-01-27
7                                            Bradford 2019-01-01 2026-01-04
8                              Bradford - City Centre 2019-01-07 2024-12-23
9                                        Bradford BID 2019-01-01 2026-01-04
10                               Draley Street Market 2019-01-07 2024-12-17
11                                        Lister Park 2019-01-07 2024-12-23
12          

In [61]:
#Check all areas in the data
df_footfall['area'].unique()

array(['Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street',
       'BD WALLS: Come on in my friend', 'Roundwood Avenue',
       'BD Walls Roots', 'BD Walls : Serving the district',
       'BD Walls - The Portal', 'BD WALL - Wayfinders', 'Bradford',
       'Bradford BID', 'Bradford - City Centre', 'Draley Street Market',
       'Lister Park', 'Baildon', 'WILD UPLANDS'], dtype=object)

The data for 2025 is missing in certain areas, thus datasets containing this needs to be integrated.

In [ ]:
import glob
import os

#Folder with containing all areas
folder_path = r"Footfall data 2025"

#Find all CSVs in all subfolders
area_folders = glob.glob(os.path.join(folder_path, '*'))

#Define file names
custom_name = ['footfall_2025']

#Dict to store combined dataframes by file type
combined = {name: pd.DataFrame() for name in custom_name}

for area_path in area_folders:
    #Get area name
    area_name = os.path.basename(area_path)
    
    csv_files = sorted(glob.glob(os.path.join(area_path, '*.csv')))
    
    #Loop over CSVs and assign new name
    for i, file in enumerate (csv_files):
        df = pd.read_csv(file, encoding='cp1252')
        
        #Add area column
        df['region'] = area_name
        
        #Get the new names
        new_name = custom_name[i]
        
        #Concatenate into 
        combined[new_name] = pd.concat([combined[new_name], df], ignore_index=True)

In [63]:
#Extract dataframe
F_2025 = combined['footfall_2025']

#Only keep columns interested in
F_2025 = F_2025[['area', 'Interval', 'Date', 'Total Footfall']]

#Keep only the daily intervals
F_2025 = F_2025[F_2025['Interval'] == 'daily']

#Drop the interval column as the dataset now only contains daily data
F_2025 = F_2025.drop(columns=['Interval'])

#Convert date column to datetime
F_2025['Date'] = pd.to_datetime(F_2025['Date'], format='mixed', dayfirst= True)
F_2025 = F_2025.rename(columns={'Date':'datestamp'})

#Rename total footfall column
F_2025 = F_2025.rename(columns={'Total Footfall': 'estimated_actual_footfall'})


#Check
F_2025.head()

,area,datestamp,estimated_actual_footfall
7,Bradford - Penistone Hill,2019-01-04,2056.59
8,Bradford - Penistone Hill,2019-01-05,4987.24
9,Bradford - Penistone Hill,2019-01-07,2493.62
10,Bradford - Penistone Hill,2019-01-11,2056.59
11,Bradford - Penistone Hill,2019-01-12,4987.24


In [64]:
#Check date ranges for each area
data_range_per_area = (
    F_2025
    .groupby('area')['datestamp']
    .agg(min_date='min', max_date='max')
    .reset_index()
)
print(data_range_per_area)

                                                 area   min_date   max_date
0       BD Walls : Root 1 Coates St, Bradford BD5 7DL 2024-12-25 2025-12-23
1                                       BD Walls RAVO 2024-12-28 2026-01-27
2             BD walls : Come on in my friend BD5 3PX 2024-12-25 2026-01-26
3                         BD walls the portal BD1 CBH 2024-12-25 2026-01-27
4   BD walls: Serving the district Morrisons, Brad... 2024-12-25 2026-01-27
5                                            Bradford 2024-12-25 2025-12-26
6                           Bradford - Penistone Hill 2019-01-04 2026-01-15
7                                Bradford city centre 2024-12-25 2026-01-27
8                              Darley's street market 2024-12-25 2026-01-27
9                                         Lister Park 2024-12-25 2026-01-27
10                    Painting the sky - Roberts part 2024-12-25 2026-01-25


In [65]:
F_2025['area'].unique()

array(['Bradford - Penistone Hill', 'Bradford',
       'BD walls : Come on in my friend BD5 3PX', 'BD Walls RAVO',
       'BD Walls : Root 1 Coates St, Bradford BD5 7DL',
       'BD walls: Serving the district Morrisons, Bradford Road, Idle BD10',
       'BD walls the portal BD1 CBH', 'Bradford city centre',
       "Darley's street market", 'Lister Park',
       'Painting the sky - Roberts part'], dtype=object)

In [66]:
df_footfall['area'].unique()

array(['Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street',
       'BD WALLS: Come on in my friend', 'Roundwood Avenue',
       'BD Walls Roots', 'BD Walls : Serving the district',
       'BD Walls - The Portal', 'BD WALL - Wayfinders', 'Bradford',
       'Bradford BID', 'Bradford - City Centre', 'Draley Street Market',
       'Lister Park', 'Baildon', 'WILD UPLANDS'], dtype=object)

In [67]:
#Rename some of the area names as they don't match between the 2019-2024 and the 2025 datasets
F_2025['area'] = F_2025['area'].replace('Bradford', 'Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street')
F_2025['area'] = F_2025['area'].replace('BD Walls RAVO', 'Roundwood Avenue')
F_2025['area'] = F_2025['area'].replace('BD walls: Serving the district Morrisons, Bradford Road, Idle BD10', 'BD Walls : Serving the district')
F_2025['area'] = F_2025['area'].replace('BD walls the portal BD1 CBH', 'BD Walls - The Portal')
F_2025['area'] = F_2025['area'].replace('Bradford city centre', 'Bradford - City Centre')
F_2025['area'] = F_2025['area'].replace("Darley's street market", 'Draley Street Market')


df_footfall['area'] = df_footfall['area'].replace('Bradford', 'Bradford BID')
df_footfall['area'] = df_footfall['area'].replace('WILD UPLANDS', 'Bradford - Penistone Hill')
df_footfall['area'] = df_footfall['area'].replace('BD WALLS: Come on in my friend', 'BD walls : Come on in my friend BD5 3PX')
df_footfall['area'] = df_footfall['area'].replace('BD Walls Roots', 'BD Walls : Root 1 Coates St, Bradford BD5 7DL')
df_footfall['area'] = df_footfall['area'].replace('Baildon', 'Painting the sky - Roberts part')

In [ ]:
#Load footfall data for the 'Bradford MetOffice' aka district area
footfall_Met = pd.read_csv(r"footfall-MetOffice.csv")
footfall_Met.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2470 entries, 0 to 2469
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   datestamp                          2470 non-null   object 
 1   centre_name                        2470 non-null   object 
 2   purchasing_power_quantile          2470 non-null   int64  
 3   estimated_actual_footfall          2306 non-null   float64
 4   estimated_actual_footfall_rolling  2470 non-null   int64  
 5   indexed_signal                     0 non-null      float64
 6   indexed_signal_rolling             0 non-null      float64
 7   source                             2470 non-null   object 
dtypes: float64(3), int64(2), object(3)
memory usage: 154.5+ KB


In [69]:
#Prepare the footfall for Bradford MetOffice area
footfall_Met = footfall_Met.drop(columns=['purchasing_power_quantile', 'indexed_signal', 'indexed_signal_rolling', 'source', 'estimated_actual_footfall_rolling'])
footfall_Met = footfall_Met.rename(columns={'centre_name': 'area'})
footfall_Met.head()

,datestamp,area,estimated_actual_footfall
0,2019-01-01,Met Office - Bradford,530996.0
1,2019-01-02,Met Office - Bradford,568621.0
2,2019-01-03,Met Office - Bradford,606939.0
3,2019-01-04,Met Office - Bradford,508695.0
4,2019-01-05,Met Office - Bradford,468546.0


In [70]:
#Add the bradford district footfall (MetOffice) and 2025 datasets
#Concatenate the now 3 datasets together (have the same columns)
footfall_mix = pd.concat([df_footfall, footfall_Met, F_2025], axis=0)
#Reset index
footfall_mix = footfall_mix.reset_index(drop= True)

#Convert datestamp to datetime
footfall_mix['datestamp'] = pd.to_datetime(footfall_mix['datestamp'])
footfall_mix.head()

,area,datestamp,estimated_actual_footfall
0,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-11,2272.41
1,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-18,2272.41
2,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-25,2272.41
3,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-27,2272.41
4,"Balancing Art - Coates Street, Bradford Foyer,...",2019-03-01,18179.26


In [71]:
#Rename all areas again for clarity and consistency
footfall_mix['area'] = footfall_mix['area'].replace('Met Office - Bradford', 'Bradford - Local Authority')
footfall_mix['area'] = footfall_mix['area'].replace('Bradford BID', 'Bradford - BID')
footfall_mix['area'] = footfall_mix['area'].replace('Lister Park', 'Bradford - Lister Park')
footfall_mix['area'] = footfall_mix['area'].replace('BD WALL - Wayfinders', 'BD Walls : Wayfinders')
footfall_mix['area'] = footfall_mix['area'].replace('BD Walls - The Portal', 'BD Walls : The Portal')
footfall_mix['area'] = footfall_mix['area'].replace('BD walls : Come on in my friend BD5 3PX', 'BD Walls : Come on in my friend')
footfall_mix['area'] = footfall_mix['area'].replace('BD Walls : Root 1 Coates St, Bradford BD5 7DL', 'BD Walls : Roots')
footfall_mix['area'] = footfall_mix['area'].replace('Roundwood Avenue', 'BD Walls: RAVO')
footfall_mix['area'] = footfall_mix['area'].replace('Draley Street Market', 'Darley Street Market')
footfall_mix['area'] = footfall_mix['area'].replace('Painting the sky - Roberts part', 'Bradford - Roberts Park')

In [72]:
footfall_mix['area'].value_counts()

area
Bradford - BID                                                               5122
Bradford - Penistone Hill                                                    3342
Bradford - City Centre                                                       2960
Bradford - Lister Park                                                       2960
BD Walls : Serving the district                                              2956
Darley Street Market                                                         2919
BD Walls : Come on in my friend                                              2910
BD Walls : The Portal                                                        2864
Bradford - Roberts Park                                                      2806
BD Walls : Wayfinders                                                        2574
BD Walls : Roots                                                             2515
Bradford - Local Authority                                                   2470
BD Walls: R

In [73]:
#Check for 0s
(footfall_mix['estimated_actual_footfall'] == 0).sum()


np.int64(0)

In [74]:
#Check for duplicates
print(footfall_mix.duplicated().sum())

#Take the average of duplicates per area per date
keys = ['datestamp', 'area']
footfall_mix = footfall_mix.groupby(keys).mean().reset_index()

#Check for duplicates again
print(footfall_mix.duplicated().sum())

39
0


In [75]:
#Check for NAs
print(footfall_mix['estimated_actual_footfall'].isna().value_counts())

#Drop missing data in 'estimated_actual_footfall'
footfall_mix = footfall_mix.dropna(subset=['estimated_actual_footfall'])
#Check again
print(footfall_mix['estimated_actual_footfall'].isna().value_counts())

estimated_actual_footfall
False    32119
True       164
Name: count, dtype: int64
estimated_actual_footfall
False    32119
Name: count, dtype: int64


In [76]:
#Remove 2026 data
footfall_mix = footfall_mix[(footfall_mix['datestamp'].dt.year) != 2026]

In [77]:
#Check date ranges per area again
data_range_per_area = (
    footfall_mix
    .groupby('area')['datestamp']
    .agg(min_date='min', max_date='max')
    .reset_index()
)
print(data_range_per_area)

                                                 area   min_date   max_date
0                     BD Walls : Come on in my friend 2019-01-10 2025-12-31
1                                    BD Walls : Roots 2019-01-11 2025-12-23
2                     BD Walls : Serving the district 2019-01-07 2025-12-31
3                               BD Walls : The Portal 2019-01-07 2025-12-30
4                               BD Walls : Wayfinders 2019-01-01 2025-12-31
5                                      BD Walls: RAVO 2019-01-28 2025-12-30
6   Balancing Art - Coates Street, Bradford Foyer,... 2019-02-11 2025-12-26
7                                      Bradford - BID 2019-01-01 2025-12-31
8                              Bradford - City Centre 2019-01-07 2025-12-31
9                              Bradford - Lister Park 2019-01-07 2025-12-31
10                         Bradford - Local Authority 2019-01-01 2025-06-21
11                          Bradford - Penistone Hill 2019-01-04 2025-12-31
12          

In [78]:
#Create year and month columns
footfall_mix['year'] = footfall_mix['datestamp'].dt.year
footfall_mix['month'] = footfall_mix['datestamp'].dt.month_name()
footfall_mix.head()

,datestamp,area,estimated_actual_footfall,year,month
0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,January
1,2019-01-01,Bradford - BID,54153.485,2019,January
2,2019-01-01,Bradford - Local Authority,530996.000,2019,January
3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,January
4,2019-01-02,Bradford - BID,158891.385,2019,January


In [79]:
#Calculate the number of missing days per year per area

#Create object with all dates between 2019 and 2026
all_dates = (
    pd.date_range(start= '2019-01-01', end='2025-12-31', freq='D')
    .to_frame(index= False, name='datestamp'))
all_dates['year'] = all_dates['datestamp'].dt.year

missing_days_counts = (
    footfall_mix
    .groupby(['area', 'year'])
    .apply(
        lambda df: (
            pd.Index(
                all_dates.loc[all_dates['year'] == df.name[1], 'datestamp'])
            .difference(pd.Index(df['datestamp']))
            .size
    ))
    .reset_index(name='footfall_missing_days')
)
print(missing_days_counts)

                               area  year  footfall_missing_days
0   BD Walls : Come on in my friend  2019                     34
1   BD Walls : Come on in my friend  2020                      7
2   BD Walls : Come on in my friend  2021                      0
3   BD Walls : Come on in my friend  2022                      0
4   BD Walls : Come on in my friend  2023                      1
..                              ...   ...                    ...
93             Darley Street Market  2021                      0
94             Darley Street Market  2022                      2
95             Darley Street Market  2023                      0
96             Darley Street Market  2024                      9
97             Darley Street Market  2025                     12

[98 rows x 3 columns]


C:\Users\qxnq723\AppData\Local\Temp\ipykernel_7652\2990790734.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [80]:
#Save missing days data to have a look
missing_days_counts = pd.DataFrame(missing_days_counts)
missing_days_counts.to_csv('missing_days_counts.csv')

In [81]:
#Get the list of missing dates for every year/location combination
missing_dates = (
    footfall_mix
    .groupby(['area', 'year'])
    .apply(
        lambda df: pd.Index(
            all_dates.loc[
                all_dates['year'] == df.name[1],
                'datestamp'
            ]
        ).difference(
            pd.Index(df['datestamp'])
        )
    )
)

C:\Users\qxnq723\AppData\Local\Temp\ipykernel_7652\976477776.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [82]:
missing_dates = pd.DataFrame(missing_dates)
missing_dates.to_csv("missing_dates.csv")

# Step 2: Create Footfall Insight CSVs - NCDO Request

In [83]:
#Reload cleaned data for every section
footfall = footfall_mix.copy()

## 1) Summary Statistics are created with the original date (with some missing days)

In [84]:
#Create table summary of footfall with important metrics for all years and all areas

def table_stats(df, missing_days_df):
    rows= []
    areas = df['area'].unique()
    years = sorted(df['year'].unique())
    
    for area in areas:
        for year in years:
            df_year = df[(df['year'] == year) & (df['area'] == area)]
        
            expected_days = 366 if pd.Timestamp(year, 12, 31).is_leap_year else 365
        
            #Get number of missing days for that year in this area
            missing = (
                missing_days_df.loc[
                    (missing_days_df['year'] == year) & (missing_days_df['area'] == area), 
                    'footfall_missing_days'
                    ].iloc[0]
                    #Check if year is in missing days of area dataframe
                    if ((missing_days_df['year'] == year) & (missing_days_df['area'] == area)).any()
                    else expected_days #if year is missing assume all days are missing
            )
            
            if not df_year.empty:
                total = df_year['estimated_actual_footfall'].sum()
                daily_avg = df_year['estimated_actual_footfall'].mean()
                high_date =   df_year.loc[df_year['estimated_actual_footfall'].idxmax(), 'datestamp']
                max_daily = df_year['estimated_actual_footfall'].max()
                low_date =  df_year.loc[df_year['estimated_actual_footfall'].idxmin(), 'datestamp']
                min_daily = df_year['estimated_actual_footfall'].min()
                high_month = (df_year.groupby('month')['estimated_actual_footfall'].sum().idxmax())
            else:
                total = daily_avg = high_date = low_date = max_daily = min_daily = high_month = pd.NA
            
        
            rows.append({
            'area': area,
            'year': year, 
            'Total_Annual_Footfall': total,
            'Daily_Avg': daily_avg,
            'Highest_Footfall_Date': high_date,
            'Max_Daily': max_daily,
            'Lowest_Footfall_Date': low_date,
            'Min_Daily': min_daily,
            'Highest_Footfall_Month': high_month,
            'Missing_days_count': missing,
            '%_Days_Observed': ((expected_days - missing) / expected_days) * 100, 
            })
    return pd.DataFrame(rows)

In [85]:
#Run function for the whole footfall data
summary_BradfordMix = table_stats(footfall, missing_days_counts)
summary_BradfordMix = pd.DataFrame(summary_BradfordMix)

summary_BradfordMix.head()

,area,year,Total_Annual_Footfall,Daily_Avg,Highest_Footfall_Date,Max_Daily,Lowest_Footfall_Date,Min_Daily,Highest_Footfall_Month,Missing_days_count,%_Days_Observed
0,BD Walls : Wayfinders,2019,8906147.17,24400.403205,2019-08-23,127621.09,2019-10-08,1901.28,August,0,100.0
1,BD Walls : Wayfinders,2020,4204969.76,11488.988415,2020-06-12,34876.06,2020-11-03,1873.73,January,0,100.0
2,BD Walls : Wayfinders,2021,4004756.08,10971.934466,2021-12-10,27673.51,2021-01-25,1637.06,November,0,100.0
3,BD Walls : Wayfinders,2022,4355503.68,11932.886795,2022-08-07,37581.38,2022-10-30,1470.34,March,0,100.0
4,BD Walls : Wayfinders,2023,4796522.21,13141.156740,2023-12-30,67077.34,2023-08-28,1286.52,March,0,100.0


In [86]:
#Save the whole table summary
summary_BradfordMix.to_csv('summary_BradfordMix.csv', index=False)

**Note 1:**
From the previous table summary, it appears that certain areas have a lot of missing days in certain years. These areas are thus removed from reporting.
The areas concerned are:
* 'Bradford - Penistone Hill' (missing around half the dates in 2019, 2022, 2023, 2024 and 2025)
* 'BD Walls: RAVO' (missing around 70% of days in 2024 and 2025)
* 'Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street' (missing 90% of 2024 and missing a lot of days over all years)

**Note 2**: The geometries of the BD walls and 'Darley Street Market' have not been provided, thus these cannot be mapped onto the NCDO dashboard map. These are thus removed from the insights datasets created for Octophin/NCDO. The areas kept, as I have geometries to map them are:
* Bradford - Local Authority
* Bradford - BID
* Bradford - City Centre
* Bradford - Lister Park
* Painting the sky - Roberts Park
* ~~Bradford - Penistone Hill (but removed because of lacking data)~~

In [87]:
#Keep only areas with enough daily data and available geometries
summary_Bradford_Mix = summary_BradfordMix[summary_BradfordMix['area'].isin(['Bradford - Local Authority',
                                                                           'Bradford - BID',
                                                                           'Bradford - City Centre',
                                                                           'Bradford - Lister Park',
                                                                           'Bradford - Roberts Park'])]

In [88]:
summary_Bradford_Mix.head()

,area,year,Total_Annual_Footfall,Daily_Avg,Highest_Footfall_Date,Max_Daily,Lowest_Footfall_Date,Min_Daily,Highest_Footfall_Month,Missing_days_count,%_Days_Observed
7,Bradford - BID,2019,3.285116e+07,90003.181301,2019-07-22,162838.175,2019-03-24,22229.695,July,0,100.0
8,Bradford - BID,2020,2.221944e+07,60708.854850,2020-02-06,142490.535,2020-05-03,12882.935,February,0,100.0
9,Bradford - BID,2021,2.446751e+07,67034.287260,2021-04-27,116080.670,2021-01-10,27236.160,October,0,100.0
10,Bradford - BID,2022,2.895879e+07,79339.147000,2022-11-21,163936.000,2022-08-12,33975.690,November,0,100.0
11,Bradford - BID,2023,3.180417e+07,87134.700356,2023-12-10,180321.540,2023-09-10,36449.660,March,0,100.0


## 2) Insights CSVs are extracted

First the average daily footfall per area per year is extracted from the table summary with the original data:

In [89]:
#Reduce decimals and round up
summary_Bradford_Mix['Daily_Avg'] = summary_Bradford_Mix['Daily_Avg'].round(1)

#Separate metric into individual CSV file
Bradford_MIX_Daily_Avg_footfall = summary_Bradford_Mix[['area', 'year', 'Daily_Avg']]
Bradford_MIX_Daily_Avg_footfall.to_csv('Bradford_Mix_Daily_Avg_footfall.csv', index=False)

C:\Users\qxnq723\AppData\Local\Temp\ipykernel_7652\1734683551.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_Bradford_Mix['Daily_Avg'] = summary_Bradford_Mix['Daily_Avg'].round(1)


Second, the total annual footfall per area per year is extracted from the table summary with original data:

In [90]:
#Reduce decimals and round up
summary_Bradford_Mix['Total_Annual_Footfall'] = summary_Bradford_Mix['Total_Annual_Footfall'].round(1)


#Separate metric
Bradford_MIX_Total_Annual_footfall = summary_Bradford_Mix[['area', 'year', 'Total_Annual_Footfall']]


#The total annual footfall for 2025 in the 'Bradford - Local Authority' is removed as that year is missing too many days
Bradford_MIX_Total_Annual_footfall = Bradford_MIX_Total_Annual_footfall[
    ~(
        (Bradford_MIX_Total_Annual_footfall['area'] == 'Bradford - Local Authority') &
        (Bradford_MIX_Total_Annual_footfall['year'] == 2025)
     )
]
#Save to CSV
Bradford_MIX_Total_Annual_footfall.to_csv('Bradford_Mix_Total_Annual_footfall.csv', index=False)

C:\Users\qxnq723\AppData\Local\Temp\ipykernel_7652\2302049253.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_Bradford_Mix['Total_Annual_Footfall'] = summary_Bradford_Mix['Total_Annual_Footfall'].round(1)


# Step 3: Create Geometry File - NCDO Request

A file containing the geometries of the areas where footfall was collected is created to allow mapping of these areas on the NCDO dashboard map.

In [91]:
summary_Bradford_Mix['area'].unique()

array(['Bradford - BID', 'Bradford - Local Authority',
       'Bradford - City Centre', 'Bradford - Lister Park',
       'Bradford - Roberts Park'], dtype=object)

In [ ]:
#Load file with different geographical areas of Bradford
GDF_areas = gpd.read_file(r"areas.geojson")
GDF_areas.head(10)

,centre_name,geometry
0,Bowling Park - BD4 7,"POLYGON ((-1.74342 53.78017, -1.74338 53.78022..."
1,Wibsey Park - BD6 3,"POLYGON ((-1.79115 53.76468, -1.79115 53.76482..."
2,Cliffe Castle Park - BD20 6,"POLYGON ((-1.91566 53.87713, -1.91694 53.87755..."
3,Lister Park - BD9 4,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,Bradford BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Met Office - Bradford,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,Bradford - City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."


In [93]:
#Rename column and names
GDF_areas = GDF_areas.rename(columns={'centre_name': 'area'})

GDF_areas['area'] = GDF_areas['area'].replace('Lister Park - BD9 4', 'Bradford - Lister Park')
GDF_areas['area'] = GDF_areas['area'].replace('Bradford BID', 'Bradford - BID')
GDF_areas['area'] = GDF_areas['area'].replace('Met Office - Bradford', 'Bradford - Local Authority')

#Check
GDF_areas.head(10)

,area,geometry
0,Bowling Park - BD4 7,"POLYGON ((-1.74342 53.78017, -1.74338 53.78022..."
1,Wibsey Park - BD6 3,"POLYGON ((-1.79115 53.76468, -1.79115 53.76482..."
2,Cliffe Castle Park - BD20 6,"POLYGON ((-1.91566 53.87713, -1.91694 53.87755..."
3,Bradford - Lister Park,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,Bradford - BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Bradford - Local Authority,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,Bradford - City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."


In [94]:
#Keep only the geometries I have footfall for
keep_areas = ['Bradford - Lister Park' , 'Bradford - BID' , 'Bradford - City Centre', 'Bradford - Local Authority', 'Bradford - Penistone Hill']
GDF_areas = GDF_areas[GDF_areas['area'].isin(keep_areas)]
#Check
GDF_areas.head(10)

,area,geometry
3,Bradford - Lister Park,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,Bradford - BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Bradford - Local Authority,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,Bradford - City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."


In [95]:
print(GDF_areas.crs)

EPSG:4326


**Important Note:** Here only the geometries provided are included. Footfall around the BD walls and Darley Street Market was collected from customised polygons within the HUQ platform which I do not have access to.

I scrap the Roberts Park geometry as it was not provided. The green spaces file was dowloaded from the [Ordnance Survey](https://osdatahub.os.uk/data/downloads/open/OpenGreenspace). From the green spaces shapefile, the Roberts Park polygon is collected.

In [ ]:
#Load file with different green spaces from Ordnance Survey
Green_spaces = gpd.read_file(r"SE_GreenspaceSite.shp")
Green_spaces.head()

,id,function,distName1,distName2,distName3,distName4,geometry
0,3D6C4AE3-840B-80C3-E063-8ECAA00A8EB3,Playing Field,None,None,None,None,"POLYGON Z ((415736.67 432338.02 0, 415750.01 4..."
1,3D6C4AE4-30C4-80C3-E063-8ECAA00A8EB3,Play Space,Hope Park,None,None,None,"POLYGON Z ((415670.18 430406.08 0, 415661.4 43..."
2,3D6C4ACC-1106-80C3-E063-8ECAA00A8EB3,Public Park Or Garden,None,None,None,None,"POLYGON Z ((415812.26 431549.17 0, 415800.98 4..."
3,3D6C4B27-670E-80C3-E063-8ECAA00A8EB3,Religious Grounds,All Saints' Church,None,None,None,"POLYGON Z ((415749.64 432057.02 0, 415712 4320..."
4,3D6C4B27-6C15-80C3-E063-8ECAA00A8EB3,Play Space,None,None,None,None,"POLYGON Z ((415727.72 431565.95 0, 415734.2 43..."


In [99]:
#Search for polygon containing Roberts Park
Green_spaces[Green_spaces['distName1'].str.contains('Roberts Park', na=False, case=False)]

,id,function,distName1,distName2,distName3,distName4,geometry
366,3D6C4B24-2FFB-80C3-E063-8ECAA00A8EB3,Public Park Or Garden,Roberts Park,None,None,None,"POLYGON Z ((413968.48 438278.07 0, 413972.6 43..."


In [100]:
#Isolate that polygon
RobertsPark = Green_spaces[Green_spaces['distName1'] == 'Roberts Park']

In [101]:
#Check CRS
print(RobertsPark.crs)

#Convert the CRS 
RobertsPark = RobertsPark.to_crs(epsg= 4326)

#Check CRS
print(RobertsPark.crs)

EPSG:27700
EPSG:4326


In [102]:
#Keep only necessary columns
RobertsPark = RobertsPark[['distName1', 'geometry']]
#Rename column
RobertsPark = RobertsPark.rename(columns={'distName1': 'area'})
#Rename park name
RobertsPark ['area'] = RobertsPark ['area'].replace('Roberts Park', 'Bradford - Roberts Park')

#Add the polygon to the geometries dataset
Bradford_Geo = pd.concat([GDF_areas, RobertsPark], axis=0)
Bradford_Geo.head(20)

,area,geometry
3,Bradford - Lister Park,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,Bradford - BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Bradford - Local Authority,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,Bradford - City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."
366,Bradford - Roberts Park,"POLYGON Z ((-1.7892 53.84057 0, -1.78914 53.84..."


In [103]:
#Create GeoJSON file with geometries for which footfall data was collected (Octophin request)
#The areas should correspond to the areas in the insights datasets provided above
Bradford_Geo.to_file('Bradford_Geo.geojson', driver='GeoJSON')

In [104]:
#Create file with just Bradford BID and City Centre geometries (bradford25 Helen request)
Bradford_Geo_2 = Bradford_Geo[Bradford_Geo['area'].isin(['Bradford - BID', 'Bradford - City Centre'])]
Bradford_Geo_2.head()
Bradford_Geo_2.to_file('Bradford_Poly.geojson', driver='GeoJSON')

# Step 4: Create Full Cleaned Dataset - NCDO Request

To allow the NCDO team to create visualisations for each of these 5 areas, there is a need to create a clean dataset with the raw footfall and full dates.

In [105]:
footfall = footfall_mix.copy()

In [106]:
#Keep only areas we created summary statistics for due to lack of daily data or lack of geometries
footfall = footfall[footfall['area'].isin(['Bradford - BID', 
                                           'Bradford - Local Authority', 
                                           'Bradford - City Centre', 
                                           'Bradford - Lister Park',
                                           'Bradford - Roberts Park'])]

#Rename columns for clarity
footfall = footfall.rename(columns={'estimated_actual_footfall': 'total_daily_footfall'})

In [107]:
#Save cleaned dataset
footfall.to_csv('footfall_cleaned.csv')